# 준비 자세(포즈) 분류 모델 학습 (Colab)

`collect_pose_data.py` 로 수집한 `pose_features.csv` 를 학습해서
정지 상태의 준비 자세 5종을 분류하는 모델을 만듭니다.

**미리 해둘 것**: `pose_features.csv` 를 Drive의
`MyDrive/hand_gesture/` 폴더에 업로드 (경로 다르면 아래 CSV_PATH 수정)

**결과물** (Jetson에 넘길 파일):
- `pose_weights.npz`
- `pose_labels.json`

특징(63개)이 이미 손 크기로 정규화되어 있어서 별도 스케일러는 쓰지 않습니다.


## 1. Drive 마운트 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd

CSV_PATH = '/content/drive/MyDrive/hand_gesture/pose_features.csv'
SAVE_DIR = '/content/drive/MyDrive/hand_gesture/model_output'

df = pd.read_csv(CSV_PATH)
print('전체 데이터 개수:', len(df))
print()
print(df['label'].value_counts())

FEATURE_COLS = [c for c in df.columns if c != 'label']
print()
print('특징 개수:', len(FEATURE_COLS))  # 63이어야 함


## 2. 전처리 및 분리

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print('학습:', X_train.shape, ' 테스트:', X_test.shape)
print('클래스 순서:', list(label_encoder.classes_))


## 3. 자세 분류 MLP 정의
이 구조는 Jetson의 `build_pose_model()` 과 완전히 동일해야 합니다.


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

pose_model = Sequential([
    Input(shape=(len(FEATURE_COLS),)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax'),
])

pose_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
pose_model.summary()


## 4. 학습

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss', patience=15,
    restore_best_weights=True, verbose=1)

history = pose_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1,
)


## 5. 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_acc = pose_model.evaluate(X_test, y_test, verbose=0)
print(f'테스트 정확도: {test_acc:.4f}')
print()

y_pred = pose_model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(
    y_test, y_pred,
    labels=np.arange(num_classes),
    target_names=label_encoder.classes_,
    zero_division=0))
print('Confusion Matrix')
print(confusion_matrix(y_test, y_pred))


## 6. Jetson용 저장 (npz + json)

In [ ]:
import os, json

os.makedirs(SAVE_DIR, exist_ok=True)

weights = pose_model.get_weights()
np.savez(os.path.join(SAVE_DIR, 'pose_weights.npz'), *weights)
print(f'가중치 배열 {len(weights)}개 저장 완료 (pose_weights.npz)')

labels = {i: name for i, name in enumerate(label_encoder.classes_)}
with open(os.path.join(SAVE_DIR, 'pose_labels.json'), 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
print('pose_labels.json 저장 완료:', labels)

print()
print('저장 위치:', SAVE_DIR)
print(os.listdir(SAVE_DIR))
print()
print('Jetson에 추가로 넘길 파일: pose_weights.npz, pose_labels.json')
